# 3D weld pool: moving frame (arc-attached)

Half-plate model for aluminium arc welding of a **monolithic plate** (joint gap not resolved).

**This model (moving / electrode-attached frame):**
- Same 3D half-domain: \(x\) along weld, \(y\) from symmetry plane, \(z=0\) irradiated surface, plate in \(z<0\)
- Window moves with the arc: **Gaussian heat flux fixed at** \(x_0\); plate streams through at \(-v_{\mathrm{weld}}\)
- Lab velocity \(\mathbf{V}\) (Darcy → 0 in solid); advection uses \((\mathbf{V}-v_{\mathrm{weld}}\mathbf{e}_x)\)
- Liquid / solid via linear liquid fraction + Darcy sink in the mushy zone
- Driving forces: **Marangoni** (\(u\) and \(v\)) + buoyancy (Boussinesq)
- **No** electromagnetism, arc pressure, free-surface deformation, or filler addition

Companion fixed-frame model: `weld_pool_3d.ipynb`.

**References** (`refs/weld_pool`):
- He, Fuerschbach, DebRoy, *J. Phys. D* 36 (2003)
- Zhao & DebRoy, *Metall. Mater. Trans. B* 32B (2001) — Al 5182, negative \(d\sigma/dT\)
- Pitscheneder et al., *Welding Journal* (1996)
- Kidess et al., *Int. J. Thermal Sci.* 104 (2016)


## Схема постановки

Половина пластины; плоскость симметрии \(y=0\) — ось шва. Облучаемая поверхность — \(z=0\); толщина уходит в \(z<0\).

Окно координат привязано к дуге: пятно нагрева неподвижно в \(x_0\), материал пластины течёт сквозь окно со скоростью \(-v_{\mathrm{weld}}\) (приток справа \(x=L_x\), выток слева \(x=0\)).

```
        q(x,y)  Gaussian, fixed at x0
              |||||
    z=0   ─────████████████████──────────── irradiated (Marangoni)
              │  liquid / mush / solid   │
    z↑        │                          │
    z=-Lz ────┴──────────────────────────┴──── bottom (convection)
         x=0 ← −v_weld  (plate) ──────────→ Lx
         outlet                         inlet T=T0

  y=0 symmetry (v=0)  ···  y = Ly (wall)
  Arc: x_c = x_0 (fixed), y_c=0 on z=0
  q_max = 3 P / (π r_b²)  →  ∫q dA = P over full plane
```

| Символ | Смысл | Типично |
|---|---|---|
| \(x\) | вдоль шва (окно с дугой) | 0…20 mm |
| \(y\) | от оси шва | 0…8 mm |
| \(z\) | от поверхности вниз | −4…0 mm |
| \(P\) | мощность дуги | ~2 kW |
| \(r_b, v\) | радиус пятна, скорость сварки | мм, мм/с |
| \(x_0\) | положение дуги в окне | ~12 mm |


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse

# Domain sketch (mm): z=0 irradiated, plate in z<0; half-plate y≥0
# Moving frame: arc fixed, plate streams −v_weld
Lx, Ly, Lz = 20.0, 8.0, 4.0
x_arc, pool_len, pool_depth, pool_half_w = 12.0, 8.0, 1.8, 3.0

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)

# --- side view x–z ---
ax = axs[0]
ax.add_patch(Rectangle((0, -Lz), Lx, Lz, facecolor="#c8d6e5", edgecolor="k", lw=1.2))
ax.add_patch(
    Rectangle(
        (x_arc - 0.5 * pool_len, -pool_depth),
        pool_len,
        pool_depth,
        facecolor="#74b9ff",
        edgecolor="k",
        lw=1,
        alpha=0.9,
    )
)
ax.annotate(
    "",
    xy=(x_arc - 5, 0.7),
    xytext=(x_arc + 2, 0.7),
    arrowprops=dict(arrowstyle="->", lw=1.6, color="C3"),
)
ax.text(x_arc - 1.5, 1.05, r"$-v_{\mathrm{weld}}$ (plate)", ha="center", color="C3", fontsize=10)
ax.plot([x_arc], [0.12], marker="^", color="C3", ms=11)
ax.text(x_arc, 1.55, r"$q$ fixed at $x_0$", ha="center", color="C3", fontsize=10)
ax.text(x_arc, -0.7 * pool_depth, "ванна", ha="center", fontsize=10)
ax.text(0.2 * Lx, -0.6 * Lz, "выток", ha="center", fontsize=9)
ax.text(0.88 * Lx, -0.6 * Lz, "приток $T_0$", ha="center", fontsize=9)
ax.annotate("", xy=(1.2, -Lz), xytext=(1.2, 0), arrowprops=dict(arrowstyle="<->", color="k", lw=1))
ax.text(0.55, -0.5 * Lz, r"$z$", ha="right", va="center", fontsize=11)
ax.text(1.45, -0.12, r"$z=0$", fontsize=9)
ax.text(1.45, -Lz - 0.2, r"$z=-L_z$", fontsize=9)
ax.annotate(
    "",
    xy=(Lx - 0.5, -Lz - 0.45),
    xytext=(1.5, -Lz - 0.45),
    arrowprops=dict(arrowstyle="->", color="k", lw=1),
)
ax.text(Lx - 0.2, -Lz - 0.45, r"$x$", va="center", fontsize=11)
ax.set_xlim(-0.5, Lx + 1.5)
ax.set_ylim(-Lz - 1.0, 2.2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(r"Вид сбоку: $x$–$z$ (дуга неподвижна)")

# --- top view x–y (half-plate) ---
ax = axs[1]
ax.add_patch(Rectangle((0, 0), Lx, Ly, facecolor="#c8d6e5", edgecolor="k", lw=1.2))
pool = Ellipse(
    (x_arc, 0),
    width=pool_len,
    height=2 * pool_half_w,
    facecolor="#74b9ff",
    edgecolor="k",
    lw=1,
    alpha=0.9,
)
ax.add_patch(pool)
pool.set_clip_path(Rectangle((0, 0), Lx, Ly, transform=ax.transData))
ax.axhline(0, color="k", ls="--", lw=1)
ax.text(Lx - 1.5, 0.35, r"$y=0$ симметрия", fontsize=9)
ax.annotate(
    "",
    xy=(x_arc - 5, Ly - 1.2),
    xytext=(x_arc + 2, Ly - 1.2),
    arrowprops=dict(arrowstyle="->", lw=1.6, color="C3"),
)
ax.text(x_arc - 1.5, Ly - 0.7, r"$-v_{\mathrm{weld}}$", ha="center", color="C3", fontsize=11)
ax.plot([x_arc], [0.15], marker="o", color="C3", ms=8, zorder=4)
ax.text(x_arc, 1.1, r"дуга $x_0$", ha="center", color="C3", fontsize=9, zorder=4)
ax.text(0.55 * Lx, 0.55 * Ly, "полупластина\n$y \\geq 0$", ha="center", fontsize=10)
ax.annotate("", xy=(Lx - 0.5, -0.6), xytext=(1.5, -0.6), arrowprops=dict(arrowstyle="->", color="k", lw=1))
ax.text(Lx - 0.2, -0.6, r"$x$", va="center", fontsize=11)
ax.annotate("", xy=(-0.5, Ly - 0.3), xytext=(-0.5, 0.3), arrowprops=dict(arrowstyle="->", color="k", lw=1))
ax.text(-0.9, 0.5 * Ly, r"$y$", ha="center", va="center", fontsize=11)
ax.set_xlim(-1.5, Lx + 1.2)
ax.set_ylim(-1.2, Ly + 1.2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(r"Вид сверху: $x$–$y$ (полудомен)")

fig.suptitle("Схема: подвижная СК (окно с электродом)", fontsize=12, y=1.02)
plt.show()


## Equations

Continuity and momentum in the arc frame. Lab velocity \(\mathbf{V}\); material derivative uses relative advection \((\mathbf{V}-v_{\mathrm{weld}}\mathbf{e}_x)\). Buoyancy along \(\mathbf{e}_z\) toward \(z=0\):

$$
\rho \partial_t \mathbf{V}
= -\rho (\mathbf{V}-v_{\mathrm{weld}}\mathbf{e}_x)\cdot\nabla\mathbf{V}
+ \nabla\cdot(\mu\nabla\mathbf{V})
- \nabla p
- C\frac{(1-f_L)^2}{f_L^3+B}\mathbf{V}
+ \rho g\beta(T-T_\mathrm{ref})\,\mathbf{e}_z
$$

Energy with apparent heat capacity in the mush:

$$
\rho c_{p,\mathrm{eff}}\partial_t T
+ \rho c_p (\mathbf{V}-v_{\mathrm{weld}}\mathbf{e}_x)\cdot\nabla T
= \nabla\cdot(k\nabla T),
\qquad
c_{p,\mathrm{eff}} = c_p + \frac{L}{T_l-T_s}\ \text{(in mush)}
$$

On \(z=0\) (irradiated face), arc fixed at \(x_0\):

$$
q = q_{\max}\exp\Big(-3\frac{(x-x_0)^2+y^2}{r_b^2}\Big)
- h(T-T_a) - \varepsilon\sigma(T^4-T_a^4)
$$

$$
\mu\partial_z u = f_L\,\frac{d\sigma}{dT}\,\partial_x T,
\quad
\mu\partial_z v = f_L\,\frac{d\sigma}{dT}\,\partial_y T,
\quad w=0
$$

Open ends: inlet \(x=L_x\) with \(T=T_0\); outlet \(x=0\) with \(\partial_x T=0\); \(u=0\) (lab), \(\partial_x v=\partial_x w=0\). On \(z=-L_z\): convection \(h_\mathrm{bottom}(T-T_a)\). Symmetry at \(y=0\): \(v=0\), insulated.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != 'heat+flow_3d':
    cand = ROOT / 'heat+flow_3d'
    if cand.is_dir():
        sys.path.insert(0, str(cand))
        MODEL_DIR = cand
    else:
        sys.path.insert(0, str(ROOT))
        MODEL_DIR = ROOT
else:
    sys.path.insert(0, str(ROOT))
    MODEL_DIR = ROOT

import importlib
import weldpool3d_moving
import weldpool3d_fast_moving

importlib.reload(weldpool3d_moving)
importlib.reload(weldpool3d_fast_moving)
from weldpool3d_moving import Material, Process, WeldPool3D
from weldpool3d_fast_moving import WeldPool3DFast

print('model dir:', MODEL_DIR)
print('solvers:', WeldPool3D.__name__, WeldPool3DFast.__name__)
print('moving frame: arc fixed, plate streams at -v_weld')


## Material (Al 5182) and process parameters

Defaults from Zhao & DebRoy (2001). Arc fixed at `x0≈12 mm` so the wake fits in the 20 mm window.  
`MESH` selects the grid only: `fast` (24×10×8), `normal` (32×12×10), `fine` (48×18×16).  
`SOLVER`: `base` = `WeldPool3D` (sparse LU), `fast` = `WeldPool3DFast` (DCT projection + 3D heat CFL).  
`t_end` ~ domain transit time \(L_x/v_{\mathrm{weld}}\) so pool metrics can plateau.


In [ ]:
MESH = "normal"  # fast | normal | fine
SOLVER = "fast"  # base | fast

GRIDS = {
    "fast":   dict(nx=24, ny=10, nz=8),
    "normal": dict(nx=32, ny=12, nz=10),
    "fine":   dict(nx=48, ny=18, nz=16),
}
if MESH not in GRIDS:
    raise ValueError(f"MESH must be one of {list(GRIDS)}")

SOLVERS = {
    "base": WeldPool3D,
    "fast": WeldPool3DFast,
}
if SOLVER not in SOLVERS:
    raise ValueError(f"SOLVER must be one of {list(SOLVERS)}")
Solver = SOLVERS[SOLVER]

mat = Material()
proc = Process(enable_flow=True, t_end=2.5, x0=0.012, **GRIDS[MESH])

model = Solver(material=mat, process=proc)
print(
    f'SOLVER={SOLVER} ({Solver.__name__}), MESH={MESH}, '
    f'grid {proc.nx}×{proc.ny}×{proc.nz}, '
    f'Domain {1e3*proc.Lx:.0f}×{1e3*proc.Ly:.0f}×{1e3*proc.Lz:.0f} mm, '
    f'P={proc.P_arc:.0f} W, q_max={proc.q_max:.2e} W/m², '
    f'x0={1e3*proc.x0:.1f} mm, v={1e3*proc.v_weld:.0f} mm/s'
)


## Run


In [ ]:
diag = model.run(record_every=0.05, progress=True, record_fields=True)
depth, length, width = model.pool_metrics()
print(
    f'final: Tmax={model.T.max():.0f} K, depth={1e3*depth:.2f} mm, '
    f'length={1e3*length:.2f} mm, width={1e3*width:.2f} mm, '
    f'heat bal={model.heat_balance_ratio():.3f}'
)
print(f'frames recorded: {len(diag.T_frames)}')


## Fields: mid-plane \(y=0\) and cross-section at fixed arc \(x_0\)


In [ ]:
u_c, v_c, w_c = model._cell_velocity()
speed = np.sqrt(u_c**2 + v_c**2 + w_c**2)

j0 = 0  # near symmetry
Xc = 1e3 * model.X[:, j0, :]
Zc = 1e3 * model.Z[:, j0, :]

ix_arc = int(np.clip(np.searchsorted(model.xc, model.arc_centre()) - 1, 0, proc.nx - 1))
Yc = 1e3 * model.Y[ix_arc, :, :]
Zc_cs = 1e3 * model.Z[ix_arc, :, :]

fig, axs = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)

im = axs[0, 0].pcolormesh(Xc, Zc, model.T[:, j0, :], shading='auto', cmap='inferno')
axs[0, 0].contour(Xc, Zc, model.T[:, j0, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
axs[0, 0].axvline(1e3 * model.arc_centre(), color='w', ls='--', lw=0.8)
fig.colorbar(im, ax=axs[0, 0], label='T [K]')
axs[0, 0].set_title(f'T mid-plane y≈0, t={model.t:.2f}s')

im = axs[0, 1].pcolormesh(Xc, Zc, model.fL[:, j0, :], shading='auto', cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=axs[0, 1], label=r'$f_L$')
axs[0, 1].set_title(r'$f_L$ mid-plane')

im = axs[0, 2].pcolormesh(Xc, Zc, speed[:, j0, :], shading='auto', cmap='viridis')
fig.colorbar(im, ax=axs[0, 2], label='|V| [m/s]')
axs[0, 2].set_title('|V| mid-plane (lab)')

im = axs[1, 0].pcolormesh(Yc, Zc_cs, model.T[ix_arc, :, :], shading='auto', cmap='inferno')
axs[1, 0].contour(Yc, Zc_cs, model.T[ix_arc, :, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
fig.colorbar(im, ax=axs[1, 0], label='T [K]')
axs[1, 0].set_title(r'T cross-section $x\approx x_0$')

im = axs[1, 1].pcolormesh(Yc, Zc_cs, model.fL[ix_arc, :, :], shading='auto', cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=axs[1, 1], label=r'$f_L$')
axs[1, 1].set_title(r'$f_L$ cross-section')

# top-view T
Xt, Yt = 1e3 * model.X[:, :, -1], 1e3 * model.Y[:, :, -1]
im = axs[1, 2].pcolormesh(Xt, Yt, model.T[:, :, -1], shading='auto', cmap='inferno')
axs[1, 2].contour(Xt, Yt, model.T[:, :, -1], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
axs[1, 2].axvline(1e3 * model.arc_centre(), color='w', ls='--', lw=0.8)
fig.colorbar(im, ax=axs[1, 2], label='T [K]')
axs[1, 2].set_title('T top surface')

for ax in axs[0, :]:
    ax.set_xlabel('x [mm]'); ax.set_ylabel('z [mm]'); ax.set_aspect('equal')
axs[1, 0].set_xlabel('y [mm]'); axs[1, 0].set_ylabel('z [mm]'); axs[1, 0].set_aspect('equal')
axs[1, 1].set_xlabel('y [mm]'); axs[1, 1].set_ylabel('z [mm]'); axs[1, 1].set_aspect('equal')
axs[1, 2].set_xlabel('x [mm]'); axs[1, 2].set_ylabel('y [mm]'); axs[1, 2].set_aspect('equal')
plt.show()


## Pool size, probe under arc, and material thermal cycle

Probe in the window is under the fixed arc (not a material-point thermocouple).  
Material-point cycle along the symmetry plane: \(t'=(x_0-x)/v_{\mathrm{weld}}\) from the longitudinal \(T(x)\) profile.


In [ ]:
t = np.array(diag.time)
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), constrained_layout=True)
axs[0].plot(t, 1e3 * np.array(diag.pool_depth), label='depth')
axs[0].plot(t, 1e3 * np.array(diag.pool_length), label='length')
axs[0].plot(t, 1e3 * np.array(diag.pool_width), label='width (full)')
axs[0].set_xlabel('t [s]'); axs[0].set_ylabel('mm'); axs[0].legend(fontsize=8)
axs[0].set_title('Pool size'); axs[0].grid(True, alpha=0.3)

axs[1].plot(t, diag.T_probe, color='C1')
axs[1].axhline(mat.T_solidus, color='c', ls='--', label=r'$T_s$')
axs[1].axhline(mat.T_liquidus, color='g', ls='--', label=r'$T_l$')
axs[1].set_xlabel('t [s]'); axs[1].set_ylabel('T [K]'); axs[1].legend(fontsize=8)
axs[1].set_title('Probe under arc (window)'); axs[1].grid(True, alpha=0.3)

axs[2].plot(t, diag.max_speed, label='|V|max')
axs[2].plot(t, diag.pe_number, label='Pe')
axs[2].set_xlabel('t [s]'); axs[2].legend(fontsize=8)
axs[2].set_title('Flow intensity'); axs[2].grid(True, alpha=0.3)
plt.show()

# Material-point thermal cycle from mid-plane top-surface profile
j0 = 0
k_top = -1
T_line = model.T[:, j0, k_top]
t_mat = (proc.x0 - model.xc) / proc.v_weld
order = np.argsort(t_mat)
fig, ax = plt.subplots(figsize=(6, 3.2), constrained_layout=True)
ax.plot(t_mat[order], T_line[order], color='C1')
ax.axhline(mat.T_solidus, color='c', ls='--', label=r'$T_s$')
ax.axhline(mat.T_liquidus, color='g', ls='--', label=r'$T_l$')
ax.axvline(0.0, color='k', ls=':', lw=0.8, label='under arc')
ax.set_xlabel(r"$t'=(x_0-x)/v_{\mathrm{weld}}$ [s]")
ax.set_ylabel('T [K]')
ax.set_title('Material-point cycle (symmetry, top)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

print(f'v=0 on symmetry: max |v(y=0)| = {np.max(np.abs(model.v[:, 0, :])):.2e} m/s')
liq = model.fL > 0.5
div = model.divergence()
print(f'max |div| in liquid: {np.max(np.abs(div[liq])) if np.any(liq) else 0:.2e} 1/s')
print(f'heat bal (incl. advective out): {model.heat_balance_ratio():.3f}')


## Animation over time

Requires `record_fields=True` in the run cell. Mid-plane \(y\approx 0\), cross-section at fixed \(x_0\), and top surface. Arc marker stays at \(x_0\).


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

assert diag.T_frames, 'No field frames — re-run with record_fields=True'

times = np.array(diag.time)
T_fr = diag.T_frames
fL_fr = diag.fL_frames
spd_fr = diag.speed_frames
x0 = model.arc_centre()
ix_arc = int(np.clip(np.searchsorted(model.xc, x0) - 1, 0, proc.nx - 1))

j0 = 0
Xc = 1e3 * model.X[:, j0, :]
Zc = 1e3 * model.Z[:, j0, :]
Xt = 1e3 * model.X[:, :, -1]
Yt = 1e3 * model.Y[:, :, -1]
Yc = 1e3 * model.Y[ix_arc, :, :]
Zcs = 1e3 * model.Z[ix_arc, :, :]

T_vmin = float(min(f.min() for f in T_fr))
T_vmax = float(max(f.max() for f in T_fr))
spd_vmax = float(max(f.max() for f in spd_fr)) or 1e-3

fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)

im0 = axs[0].pcolormesh(Xc, Zc, T_fr[0][:, j0, :], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
axs[0].axvline(1e3 * x0, color='w', ls='--', lw=0.8)
fig.colorbar(im0, ax=axs[0], label='T [K]')
axs[0].set_xlabel('x [mm]'); axs[0].set_ylabel('z [mm]'); axs[0].set_aspect('equal')
title0 = axs[0].set_title(f'T mid-plane, t={times[0]:.2f}s')

im1 = axs[1].pcolormesh(Yc, Zcs, T_fr[0][ix_arc, :, :], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
fig.colorbar(im1, ax=axs[1], label='T [K]')
axs[1].set_xlabel('y [mm]'); axs[1].set_ylabel('z [mm]'); axs[1].set_aspect('equal')
title1 = axs[1].set_title(r'T cross-section $x\approx x_0$')

im2 = axs[2].pcolormesh(Xt, Yt, T_fr[0][:, :, -1], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
axs[2].axvline(1e3 * x0, color='w', ls='--', lw=0.8)
fig.colorbar(im2, ax=axs[2], label='T [K]')
axs[2].set_xlabel('x [mm]'); axs[2].set_ylabel('y [mm]'); axs[2].set_aspect('equal')
title2 = axs[2].set_title('T top surface')


def update(i):
    Ti = T_fr[i]
    im0.set_array(Ti[:, j0, :].ravel())
    title0.set_text(f'T mid-plane, t={times[i]:.2f}s')
    im1.set_array(Ti[ix_arc, :, :].ravel())
    title1.set_text(rf'T cross-section $x_0$={1e3*x0:.1f} mm')
    im2.set_array(Ti[:, :, -1].ravel())
    title2.set_text(f'T top, t={times[i]:.2f}s')
    for ax in axs:
        for coll in list(ax.collections):
            if coll not in (im0, im1, im2):
                try:
                    coll.remove()
                except ValueError:
                    pass
    axs[0].contour(Xc, Zc, Ti[:, j0, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
    axs[1].contour(Yc, Zcs, Ti[ix_arc, :, :], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
    axs[2].contour(Xt, Yt, Ti[:, :, -1], levels=[mat.T_solidus, mat.T_liquidus], colors=['c', 'lime'])
    return (im0, im1, im2, title0, title1, title2)


anim_T = FuncAnimation(fig, update, frames=len(T_fr), interval=150, blit=False)
plt.close(fig)
HTML(anim_T.to_jshtml())


In [ ]:
# f_L mid-plane + |V| mid-plane + pool size curves with time cursor
fig2, axs2 = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)

im_f = axs2[0].pcolormesh(Xc, Zc, fL_fr[0][:, j0, :], shading='auto', cmap='Blues', vmin=0, vmax=1)
fig2.colorbar(im_f, ax=axs2[0], label=r'$f_L$')
axs2[0].set_xlabel('x [mm]'); axs2[0].set_ylabel('z [mm]'); axs2[0].set_aspect('equal')
t_f = axs2[0].set_title(rf'$f_L$ mid-plane, t={times[0]:.2f}s')

im_s = axs2[1].pcolormesh(Xc, Zc, spd_fr[0][:, j0, :], shading='auto', cmap='viridis', vmin=0, vmax=spd_vmax)
fig2.colorbar(im_s, ax=axs2[1], label='|V| [m/s]')
axs2[1].set_xlabel('x [mm]'); axs2[1].set_ylabel('z [mm]'); axs2[1].set_aspect('equal')
t_s = axs2[1].set_title(f'|V| mid-plane, t={times[0]:.2f}s')

axs2[2].plot(times, 1e3 * np.array(diag.pool_depth), label='depth')
axs2[2].plot(times, 1e3 * np.array(diag.pool_length), label='length')
axs2[2].plot(times, 1e3 * np.array(diag.pool_width), label='width')
cursor = axs2[2].axvline(times[0], color='k', ls='--', lw=1)
axs2[2].set_xlabel('t [s]'); axs2[2].set_ylabel('mm')
axs2[2].legend(fontsize=8); axs2[2].grid(True, alpha=0.3)
axs2[2].set_title('Pool size')


def update2(i):
    im_f.set_array(fL_fr[i][:, j0, :].ravel())
    im_s.set_array(spd_fr[i][:, j0, :].ravel())
    t_f.set_text(rf'$f_L$ mid-plane, t={times[i]:.2f}s')
    t_s.set_text(f'|V| mid-plane, t={times[i]:.2f}s')
    cursor.set_xdata([times[i], times[i]])
    return (im_f, im_s, cursor, t_f, t_s)


anim_flow = FuncAnimation(fig2, update2, frames=len(fL_fr), interval=150, blit=False)
plt.close(fig2)
HTML(anim_flow.to_jshtml())


## Flow animation: mid-plane $x$–$z$ and cross-section $y$–$z$

Requires `record_fields=True` (velocity components are stored with the other field frames). Left: $y\approx 0$ with arrows $(u,w)$. Right: slice at fixed arc $x\approx x_0$ with arrows $(v,w)$. Arrows only in the liquid ($f_L>0.3$).


In [ ]:
assert diag.T_frames, 'No field frames — re-run with record_fields=True'
assert diag.u_frames, 'No velocity frames — re-run after updating weldpool3d (u/v/w recording)'

times = np.array(diag.time)
spd_fr = diag.speed_frames
fL_fr = diag.fL_frames
u_fr = diag.u_frames
v_fr = diag.v_frames
w_fr = diag.w_frames
x0 = model.arc_centre()
ix_arc = int(np.clip(np.searchsorted(model.xc, x0) - 1, 0, proc.nx - 1))

j0 = 0
Xc = 1e3 * model.X[:, j0, :]
Zc = 1e3 * model.Z[:, j0, :]

spd_vmax = float(max(f.max() for f in spd_fr)) or 1e-3
sx = max(1, proc.nx // 16)
sy = max(1, proc.ny // 8)
sz = max(1, proc.nz // 6)

# Global quiver scale from peak speed in the liquid across all frames
vmax_liq = 1e-9
for i in range(len(spd_fr)):
    mask = fL_fr[i] > 0.3
    if np.any(mask):
        vmax_liq = max(vmax_liq, float(np.max(spd_fr[i][mask])))
q_scale = vmax_liq / 0.12


def _xz_quiver_data(i):
    mask = fL_fr[i][:, j0, :] > 0.3
    ms = mask[::sx, ::sz]
    Xs = Xc[::sx, ::sz][ms]
    Zs = Zc[::sx, ::sz][ms]
    Us = u_fr[i][::sx, j0, ::sz][ms]
    Ws = w_fr[i][::sx, j0, ::sz][ms]
    if Xs.size == 0:
        return np.array([np.nan]), np.array([np.nan]), np.array([0.0]), np.array([0.0])
    return Xs, Zs, Us, Ws


def _yz_quiver_data(i, ix):
    Ymm = 1e3 * model.Y[ix, :, :]
    Zmm = 1e3 * model.Z[ix, :, :]
    mask = fL_fr[i][ix, :, :] > 0.3
    ms = mask[::sy, ::sz]
    Ys = Ymm[::sy, ::sz][ms]
    Zs = Zmm[::sy, ::sz][ms]
    Vs = v_fr[i][ix, ::sy, ::sz][ms]
    Ws = w_fr[i][ix, ::sy, ::sz][ms]
    if Ys.size == 0:
        return np.array([np.nan]), np.array([np.nan]), np.array([0.0]), np.array([0.0])
    return Ys, Zs, Vs, Ws


fig3, axs3 = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)

im_xz = axs3[0].pcolormesh(
    Xc, Zc, spd_fr[0][:, j0, :], shading='auto', cmap='viridis', vmin=0, vmax=spd_vmax
)
axs3[0].contour(
    Xc, Zc, fL_fr[0][:, j0, :], levels=[0.99], colors=['cyan'], linewidths=1.0
)
arc_line_v = axs3[0].axvline(1e3 * x0, color='w', ls='--', lw=0.8)
Xs0, Zs0, Us0, Ws0 = _xz_quiver_data(0)
q_xz = axs3[0].quiver(
    Xs0, Zs0, Us0, Ws0,
    color='w', angles='uv', scale_units='height', scale=q_scale,
    width=0.004, pivot='mid',
)
fig3.colorbar(im_xz, ax=axs3[0], label='|V| [m/s]')
axs3[0].set_xlabel('x [mm]'); axs3[0].set_ylabel('z [mm]'); axs3[0].set_aspect('equal')
title_xz = axs3[0].set_title(f'Flow x–z (y≈0), t={times[0]:.2f}s')
axs3[0].quiverkey(q_xz, 0.82, 1.08, 0.2, '0.2 m/s', labelpos='E', coordinates='axes', color='k')

ix0 = ix_arc
Yc0 = 1e3 * model.Y[ix0, :, :]
Zcs0 = 1e3 * model.Z[ix0, :, :]
im_yz = axs3[1].pcolormesh(
    Yc0, Zcs0, spd_fr[0][ix0, :, :], shading='auto', cmap='viridis', vmin=0, vmax=spd_vmax
)
axs3[1].contour(
    Yc0, Zcs0, fL_fr[0][ix0, :, :], levels=[0.99], colors=['cyan'], linewidths=1.0
)
Ys0, Zs0y, Vs0, Ws0y = _yz_quiver_data(0, ix0)
q_yz = axs3[1].quiver(
    Ys0, Zs0y, Vs0, Ws0y,
    color='w', angles='uv', scale_units='height', scale=q_scale,
    width=0.004, pivot='mid',
)
fig3.colorbar(im_yz, ax=axs3[1], label='|V| [m/s]')
axs3[1].set_xlabel('y [mm]'); axs3[1].set_ylabel('z [mm]'); axs3[1].set_aspect('equal')
title_yz = axs3[1].set_title(rf'Flow y–z at $x\approx x_c$')


def update_flow_planes(i):
    global q_xz, q_yz
    q_xz.remove()
    q_yz.remove()

    im_xz.set_array(spd_fr[i][:, j0, :].ravel())
    pass  # arc fixed at x0
    title_xz.set_text(f'Flow x–z (y≈0), t={times[i]:.2f}s')

    ix = ix_arc
    Yi = 1e3 * model.Y[ix, :, :]
    Zi = 1e3 * model.Z[ix, :, :]
    im_yz.set_array(spd_fr[i][ix, :, :].ravel())
    title_yz.set_text(f'Flow y–z at x0={1e3*x0:.1f} mm')

    for ax in axs3:
        for coll in list(ax.collections):
            if coll not in (im_xz, im_yz):
                try:
                    coll.remove()
                except ValueError:
                    pass
    axs3[0].contour(Xc, Zc, fL_fr[i][:, j0, :], levels=[0.99], colors=['cyan'], linewidths=1.0)
    axs3[1].contour(Yi, Zi, fL_fr[i][ix, :, :], levels=[0.99], colors=['cyan'], linewidths=1.0)

    Xs, Zs, Us, Ws = _xz_quiver_data(i)
    q_xz = axs3[0].quiver(
        Xs, Zs, Us, Ws,
        color='w', angles='uv', scale_units='height', scale=q_scale,
        width=0.004, pivot='mid',
    )
    Ys, Zsy, Vs, Wsy = _yz_quiver_data(i, ix)
    q_yz = axs3[1].quiver(
        Ys, Zsy, Vs, Wsy,
        color='w', angles='uv', scale_units='height', scale=q_scale,
        width=0.004, pivot='mid',
    )
    return (im_xz, im_yz, arc_line_v, title_xz, title_yz)


anim_vel = FuncAnimation(fig3, update_flow_planes, frames=len(spd_fr), interval=150, blit=False)
plt.close(fig3)
HTML(anim_vel.to_jshtml())


## Checks

- Heat balance `|Q_in − Q_loss − Q_adv − ΔH| / Q_in` (includes enthalpy leaving with the plate)
- \(v=0\) on the symmetry plane
- Divergence after projection is machine-zero in the liquid
- For \(d\sigma/dT < 0\) surface flow leaves the hot spot
- Arc centre stays at \(x_0\)
